# Optimization Algorithms — From Scratch

**Module 00 — Mathematical Foundations for AI**

Every neural network is trained by an optimization algorithm. This notebook implements
and visualizes the major optimizers used in modern AI.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from typing import Callable, List, Tuple

np.random.seed(42)

## 1. Loss Surfaces

Define test functions to visualize optimizer behavior.

In [ ]:
# Quadratic (convex - has unique minimum)
def quadratic(w: np.ndarray) -> float:
    return 0.5 * w[0]**2 + 2.5 * w[1]**2

def quadratic_grad(w: np.ndarray) -> np.ndarray:
    return np.array([w[0], 5 * w[1]])

# Rosenbrock (non-convex, narrow valley - hard for optimizers)
def rosenbrock(w: np.ndarray) -> float:
    return (1 - w[0])**2 + 100 * (w[1] - w[0]**2)**2

def rosenbrock_grad(w: np.ndarray) -> np.ndarray:
    dx = -2*(1 - w[0]) + 200*(w[1] - w[0]**2)*(-2*w[0])
    dy = 200*(w[1] - w[0]**2)
    return np.array([dx, dy])

# Beale (multiple saddle points)
def beale(w: np.ndarray) -> float:
    x, y = w
    return (1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2

def beale_grad(w: np.ndarray) -> np.ndarray:
    x, y = w
    dx = 2*(1.5 - x + x*y)*(y - 1) + 2*(2.25 - x + x*y**2)*(y**2 - 1) + 2*(2.625 - x + x*y**3)*(y**3 - 1)
    dy = 2*(1.5 - x + x*y)*x + 2*(2.25 - x + x*y**2)*(2*x*y) + 2*(2.625 - x + x*y**3)*(3*x*y**2)
    return np.array([dx, dy])

## 2. Implementing Optimizers from Scratch

In [ ]:
class VanillaGD:
    """Vanilla Gradient Descent.
    
    w_{t+1} = w_t - η ∇L(w_t)
    
    Pros: Simple, guaranteed convergence for convex functions
    Cons: Slow, sensitive to learning rate, gets stuck in saddle points
    """
    def __init__(self, lr: float = 0.01):
        self.lr = lr
    
    def step(self, w: np.ndarray, grad: np.ndarray) -> np.ndarray:
        return w - self.lr * grad


class MomentumGD:
    """SGD with Momentum.
    
    v_{t+1} = β v_t + (1-β) ∇L
    w_{t+1} = w_t - η v_{t+1}
    
    Analogy: A ball rolling down a hill accumulates velocity.
    Helps escape local minima and traverse flat regions.
    """
    def __init__(self, lr: float = 0.01, beta: float = 0.9):
        self.lr = lr
        self.beta = beta
        self.velocity = None
    
    def step(self, w: np.ndarray, grad: np.ndarray) -> np.ndarray:
        if self.velocity is None:
            self.velocity = np.zeros_like(w)
        self.velocity = self.beta * self.velocity + (1 - self.beta) * grad
        return w - self.lr * self.velocity


class RMSProp:
    """RMSProp: Adaptive learning rate per parameter.
    
    s_t = β s_{t-1} + (1-β) g_t²
    w_t = w_{t-1} - η / (√s_t + ε) * g_t
    
    Key insight: Divide by running average of squared gradients.
    Parameters with large gradients get smaller updates.
    Parameters with small gradients get larger updates.
    """
    def __init__(self, lr: float = 0.01, beta: float = 0.999, eps: float = 1e-8):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.sq_grad_avg = None
    
    def step(self, w: np.ndarray, grad: np.ndarray) -> np.ndarray:
        if self.sq_grad_avg is None:
            self.sq_grad_avg = np.zeros_like(w)
        self.sq_grad_avg = self.beta * self.sq_grad_avg + (1 - self.beta) * grad**2
        return w - self.lr * grad / (np.sqrt(self.sq_grad_avg) + self.eps)


class Adam:
    """Adam: Adaptive Moment Estimation.
    
    Combines Momentum + RMSProp + Bias Correction.
    
    m_t = β₁ m_{t-1} + (1-β₁) g_t           (first moment / mean)
    v_t = β₂ v_{t-1} + (1-β₂) g_t²           (second moment / variance)
    m̂_t = m_t / (1 - β₁ᵗ)                    (bias correction)
    v̂_t = v_t / (1 - β₂ᵗ)                    (bias correction)
    w_t = w_{t-1} - η m̂_t / (√v̂_t + ε)
    
    This is THE default optimizer for training transformers.
    """
    def __init__(self, lr: float = 0.001, beta1: float = 0.9, 
                 beta2: float = 0.999, eps: float = 1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None  # first moment
        self.v = None  # second moment
        self.t = 0     # time step
    
    def step(self, w: np.ndarray, grad: np.ndarray) -> np.ndarray:
        if self.m is None:
            self.m = np.zeros_like(w)
            self.v = np.zeros_like(w)
        
        self.t += 1
        
        # Update biased first moment estimate
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        # Update biased second moment estimate
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        
        # Bias correction (critical for early steps!)
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class AdamW:
    """AdamW: Adam with Decoupled Weight Decay.
    
    The key difference from Adam + L2:
    - Adam + L2: regularization is scaled by adaptive learning rate (bad!)
    - AdamW: weight decay is applied directly to weights (correct!)
    
    w_t = w_{t-1} - η (m̂_t / (√v̂_t + ε) + λ w_{t-1})
    
    This is the standard for training LLMs (GPT, LLaMA, etc.)
    """
    def __init__(self, lr: float = 0.001, beta1: float = 0.9, 
                 beta2: float = 0.999, eps: float = 1e-8, weight_decay: float = 0.01):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = None
        self.v = None
        self.t = 0
    
    def step(self, w: np.ndarray, grad: np.ndarray) -> np.ndarray:
        if self.m is None:
            self.m = np.zeros_like(w)
            self.v = np.zeros_like(w)
        
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        # Decoupled weight decay: applied to weights, NOT gradients
        return w - self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.weight_decay * w)

## 3. Optimizer Comparison on Quadratic Surface

In [ ]:
def run_optimizer(optimizer, grad_fn, w_init, n_steps=200):
    """Run optimizer and record trajectory."""
    w = w_init.copy()
    trajectory = [w.copy()]
    for _ in range(n_steps):
        grad = grad_fn(w)
        w = optimizer.step(w, grad)
        trajectory.append(w.copy())
    return np.array(trajectory)

# Starting point
w_init = np.array([4.0, 4.0])

# Create optimizers
optimizers = {
    'Vanilla GD (lr=0.1)': VanillaGD(lr=0.1),
    'Momentum (lr=0.1)': MomentumGD(lr=0.1, beta=0.9),
    'RMSProp (lr=0.1)': RMSProp(lr=0.1),
    'Adam (lr=0.1)': Adam(lr=0.1),
}

# Run all optimizers
trajectories = {}
for name, opt in optimizers.items():
    trajectories[name] = run_optimizer(opt, quadratic_grad, w_init, n_steps=50)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Contour plot
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = 0.5 * X**2 + 2.5 * Y**2

colors = ['red', 'blue', 'green', 'purple']
for (name, traj), color in zip(trajectories.items(), colors):
    axes[0].contour(X, Y, Z, levels=20, alpha=0.3)
    axes[0].plot(traj[:, 0], traj[:, 1], '-o', label=name, color=color, 
                markersize=3, linewidth=1.5)

axes[0].plot(0, 0, 'k*', markersize=15, label='Minimum')
axes[0].set_title('Optimizer Trajectories on Quadratic Surface')
axes[0].set_xlabel('w₁')
axes[0].set_ylabel('w₂')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Loss curves
for (name, traj), color in zip(trajectories.items(), colors):
    losses = [quadratic(w) for w in traj]
    axes[1].plot(losses, label=name, color=color, linewidth=2)

axes[1].set_title('Loss vs. Iteration')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../diagrams/optimizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Learning Rate Schedules

Modern training uses learning rate schedules:
- **Warmup**: Start with small lr, linearly increase
- **Cosine decay**: Smoothly decrease lr following cosine curve
- **Linear decay**: Linearly decrease lr to 0

In [ ]:
def cosine_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0):
    """Cosine learning rate schedule with warmup.
    
    Used by: GPT-3, LLaMA, most modern LLMs.
    """
    if step < warmup_steps:
        # Linear warmup
        return max_lr * step / warmup_steps
    else:
        # Cosine decay
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr + 0.5 * (max_lr - min_lr) * (1 + np.cos(np.pi * progress))

# Visualize schedule
total_steps = 10000
warmup_steps = 1000
max_lr = 3e-4
min_lr = 1e-5

steps = range(total_steps)
lrs = [cosine_schedule(s, total_steps, warmup_steps, max_lr, min_lr) for s in steps]

plt.figure(figsize=(12, 5))
plt.plot(steps, lrs, linewidth=2)
plt.axvline(x=warmup_steps, color='r', linestyle='--', label=f'End of warmup ({warmup_steps} steps)')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Learning Rate Schedule with Warmup (Standard for LLM Training)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../diagrams/lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Max LR: {max_lr}")
print(f"Min LR: {min_lr}")
print(f"Warmup: {warmup_steps} steps ({warmup_steps/total_steps*100:.0f}%)")

## 5. Gradient Descent for Linear Regression

Let's use our optimizers to train a real model: linear regression.

In [ ]:
# Generate synthetic data
n_samples = 100
X = np.random.randn(n_samples, 2)  # 2 features
true_w = np.array([3.0, -2.0])
true_b = 1.5
y = X @ true_w + true_b + np.random.randn(n_samples) * 0.5

# Loss function: MSE = (1/n) Σ (y - ŷ)²
def mse_loss(X, y, w, b):
    predictions = X @ w + b
    return np.mean((y - predictions) ** 2)

def mse_gradient(X, y, w, b):
    n = len(y)
    predictions = X @ w + b
    residuals = predictions - y
    dw = (2/n) * X.T @ residuals
    db = (2/n) * np.sum(residuals)
    return dw, db

# Train with Adam
w = np.zeros(2)
b = 0.0
lr = 0.1

adam_w = Adam(lr=lr)
adam_b = Adam(lr=lr)

losses = []
for step in range(200):
    loss = mse_loss(X, y, w, b)
    losses.append(loss)
    
    dw, db = mse_gradient(X, y, w, b)
    w = adam_w.step(w, dw)
    b_arr = adam_b.step(np.array([b]), np.array([db]))
    b = b_arr[0]
    
    if step % 50 == 0:
        print(f"Step {step:3d}: loss={loss:.4f}, w={w}, b={b:.4f}")

print(f"\nLearned:    w={w}, b={b:.4f}")
print(f"True:       w={true_w}, b={true_b}")

plt.figure(figsize=(10, 5))
plt.plot(losses, linewidth=2)
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('Training Linear Regression with Adam')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

## Key Takeaways

| Optimizer | When to Use | Key Property |
|-----------|------------|---------------|
| SGD | Convex problems, when you need guarantees | Simple, well-understood |
| SGD + Momentum | Computer vision (ResNet, etc.) | Faster convergence, escapes local minima |
| Adam | Default for most tasks | Adaptive per-parameter learning rates |
| AdamW | **LLM training standard** | Proper weight decay for generalization |

**In practice**:
- LLMs: AdamW with cosine schedule, warmup = 1-5% of training, max_lr ≈ 3e-4 to 1e-3
- Fine-tuning: AdamW with lower lr (1e-5 to 5e-5)
- CNN training: SGD + Momentum often works better than Adam